# IMDB Sentiment Classification: Embedding + LSTM

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
results_dir = project_root / "results"
models_dir = project_root / "models"
results_dir.mkdir(exist_ok=True)
models_dir.mkdir(exist_ok=True)

VOCAB_SIZE = 10000
MAX_LEN = 200

print("Imports ready")
print("Project root:", project_root)
print("VOCAB_SIZE:", VOCAB_SIZE, "MAX_LEN:", MAX_LEN)

Matplotlib is building the font cache; this may take a moment.


Imports ready
Project root: /Users/yuvrajgole/Documents/DL Experiments/DL Experiment 5/nlp-imdb-lstm
VOCAB_SIZE: 10000 MAX_LEN: 200


## Section 1: Load Dataset

In [2]:
(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)

print("len(X_train):", len(X_train))
print("len(X_test):", len(X_test))

       0/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

   32768/17464789 ━━━━━━━━━━━━━━━━━━━━ 33s 2us/step

   49152/17464789 ━━━━━━━━━━━━━━━━━━━━ 1:42 6us/step

   81920/17464789 ━━━━━━━━━━━━━━━━━━━━ 1:54 7us/step

  114688/17464789 ━━━━━━━━━━━━━━━━━━━━ 1:31 5us/step

...

15646720/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

16318464/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

17006592/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


len(X_train): 25000
len(X_test): 25000


## Section 2: Text Preprocessing (Padding)

In [3]:
X_train = pad_sequences(X_train, maxlen=MAX_LEN, padding="post", truncating="post")
X_test = pad_sequences(X_test, maxlen=MAX_LEN, padding="post", truncating="post")

print("X_train.shape:", X_train.shape)
print("X_test.shape:", X_test.shape)

X_train.shape: (25000, 200)
X_test.shape: (25000, 200)


## Section 3: Build Embedding + LSTM Model

In [4]:
model = keras.Sequential(
    [
        keras.layers.Input(shape=(MAX_LEN,)),
        keras.layers.Embedding(VOCAB_SIZE, 128),
        keras.layers.LSTM(64, dropout=0.2, recurrent_dropout=0.2),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation="sigmoid"),
    ]
)
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,331,521 (5.08 MB)

 Trainable params: 1,331,521 (5.08 MB)

 Non-trainable params: 0 (0.00 B)

## Section 4: Train the Model

In [5]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=64,
    verbose=1,
)

Epoch 1/5


  1/313 ━━━━━━━━━━━━━━━━━━━━ 8:13 2s/step - accuracy: 0.4688 - loss: 0.6929

  3/313 ━━━━━━━━━━━━━━━━━━━━ 13s 43ms/step - accuracy: 0.5156 - loss: 0.6914

  5/313 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.5094 - loss: 0.6896

  7/313 ━━━━━━━━━━━━━━━━━━━━ 12s 42ms/step - accuracy: 0.4911 - loss: 0.6929

...

306/313 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8337 - loss: 0.4006

308/313 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8335 - loss: 0.4009

310/313 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8337 - loss: 0.4005

312/313 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8334 - loss: 0.4008

313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 44ms/step - accuracy: 0.8333 - loss: 0.4011 - val_accuracy: 0.7818 - val_loss: 0.4575


## Section 5: Evaluate with Precision, Recall, F1-score

In [6]:
y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(cm)

print("\nClassification report:")
print(
    classification_report(
        y_test, y_pred, target_names=["Negative", "Positive"]
    )
)

report_dict = classification_report(
    y_test,
    y_pred,
    target_names=["Negative", "Positive"],
    output_dict=True,
)
report_df = pd.DataFrame(report_dict).transpose()
report_csv = results_dir / "classification_report.csv"
report_df.to_csv(report_csv)
print("Saved:", report_csv)

overall_test_acc = float((y_pred == y_test).mean())
print(f"Overall Test Accuracy: {overall_test_acc * 100:.2f}%")

Confusion matrix:
[[ 9077  3423]
 [ 2182 10318]]

Classification report:
              precision    recall  f1-score   support

    Negative       0.81      0.73      0.76     12500
    Positive       0.75      0.83      0.79     12500

    accuracy                           0.78     25000
   macro avg       0.78      0.78      0.78     25000
weighted avg       0.78      0.78      0.78     25000

Saved: /Users/yuvrajgole/Documents/DL Experiments/DL Experiment 5/nlp-imdb-lstm/results/classification_report.csv
Overall Test Accuracy: 77.58%


## Section 6: Visualize Results

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle("Training vs Validation Curves")
fig.tight_layout()
curves_path = results_dir / "training_curves.png"
fig.savefig(curves_path, dpi=120, bbox_inches="tight")
plt.close(fig)
print("Saved:", curves_path)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"],
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
fig.tight_layout()
cm_path = results_dir / "confusion_matrix.png"
fig.savefig(cm_path, dpi=120, bbox_inches="tight")
plt.close(fig)
print("Saved:", cm_path)

Saved: /Users/yuvrajgole/Documents/DL Experiments/DL Experiment 5/nlp-imdb-lstm/results/training_curves.png


Saved: /Users/yuvrajgole/Documents/DL Experiments/DL Experiment 5/nlp-imdb-lstm/results/confusion_matrix.png


## Section 7: Save Model

In [8]:
model_path = models_dir / "imdb_lstm.keras"
model.save(model_path)

print(f"Saved model to {model_path} (Keras .keras format)")
print("Text Preprocessed and Padded")
print("Embedding + LSTM Model Trained")
print(f"Overall Test Accuracy: {overall_test_acc * 100:.2f}%")
print("Evaluation Metrics Generated")

Saved model to /Users/yuvrajgole/Documents/DL Experiments/DL Experiment 5/nlp-imdb-lstm/models/imdb_lstm.keras (Keras .keras format)
Text Preprocessed and Padded
Embedding + LSTM Model Trained
Overall Test Accuracy: 77.58%
Evaluation Metrics Generated


## Section 8: Results Summary

In [9]:
summary_table = pd.DataFrame(
    {
        "Class": ["Negative", "Positive"],
        "Precision": [
            report_dict["Negative"]["precision"],
            report_dict["Positive"]["precision"],
        ],
        "Recall": [
            report_dict["Negative"]["recall"],
            report_dict["Positive"]["recall"],
        ],
        "F1-Score": [
            report_dict["Negative"]["f1-score"],
            report_dict["Positive"]["f1-score"],
        ],
    }
)
print(summary_table.to_string(index=False))
summary_table

   Class  Precision  Recall  F1-Score
Negative   0.806199 0.72616  0.764089
Positive   0.750891 0.82544  0.786403


,Class,Precision,Recall,F1-Score
0,Negative,0.806199,0.72616,0.764089
1,Positive,0.750891,0.82544,0.786403


### Completion Checklist

- [x] Text Preprocessed and Padded
- [x] Embedding + LSTM Model Trained
- [x] Overall Test Accuracy: 77.58%
- [x] Evaluation Metrics Generated